# Data Science for Business - Smoothing Splines and GAMs on Ames Housing

*Session 6 · ISLP 7.5 Smoothing Splines · 7.7 Generalized Additive Models*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install "pygam>=0.10"

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error

import statsmodels.formula.api as smf

from pygam import LinearGAM, s

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_06/ameshousing.csv')

## Prepare data

`MSSubClass` (type of dwelling) is stored as a number, but its values are codes for categories (e.g., 20 = 1-story house built 1946 or later, 60 = 2-story house built 1946 or later; see the data description). We therefore add the prefix `MSSC` to all values, which turns the variable into a string, so that it is treated as categorical.

In [ ]:
data["MSSubClass"] = "MSSC" + data["MSSubClass"].astype(str)

Let us focus on the numerical variables only (this also excludes `MSSubClass`, which is now categorical).

In [ ]:
data = data.select_dtypes(include='number')

Finally, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

## Smoothing splines with one predictor

In the notebook `ames_poly_steps_and_splines`, we chose the knots of regression splines ourselves. A **smoothing spline** (ISLP 7.5) avoids this: it puts a knot at (almost) every value of the predictor and controls the wiggliness with a **penalty λ** instead. It looks for the curve that fits the training data well, but pays a price for every wiggle (ISLP 7.5.1):

- λ close to 0: hardly any penalty, so the curve can wiggle and follow every bump of the training data.
- Very large λ: wiggles are expensive, so the curve becomes (almost) a straight line.

Smoothing splines are the building blocks of the GAM later in this notebook. We first look at a single one, again for the year of construction (`YearBuilt`).

We use the package *pygam*. Strictly speaking, pygam approximates a smoothing spline: instead of a knot at every observation, it uses a fixed number of cubic spline functions plus the penalty. `s(0)` means "a smooth function of the first column"; pygam refers to predictors by their column position, so we pass a data frame that contains only `YearBuilt`.

We fit three smoothing splines with hand-picked penalties: a very small, a medium and a very large λ (`lam`).

In [ ]:
smooth_small_lam = LinearGAM(s(0), lam=0.01).fit(X_train[['YearBuilt']], y_train)
smooth_medium_lam = LinearGAM(s(0), lam=100).fit(X_train[['YearBuilt']], y_train)
smooth_large_lam = LinearGAM(s(0), lam=1000000).fit(X_train[['YearBuilt']], y_train)

Similar to ISLP Figure 7.8, we draw the three curves into one plot. To draw them, we need a grid of years, from the oldest to the newest house in the training data.

In [ ]:
year_grid = pd.DataFrame({'YearBuilt': np.arange(X_train['YearBuilt'].min(), X_train['YearBuilt'].max() + 1)})

plt.figure(figsize=(8, 5))
plt.scatter(X_train['YearBuilt'], y_train, s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], smooth_small_lam.predict(year_grid), color='red', linewidth=2, label='λ = 0.01')
plt.plot(year_grid['YearBuilt'], smooth_medium_lam.predict(year_grid), color='blue', linewidth=2, label='λ = 100')
plt.plot(year_grid['YearBuilt'], smooth_large_lam.predict(year_grid), color='green', linewidth=2, label='λ = 1,000,000')
plt.legend()
plt.title('Smoothing splines')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

With the small λ (red), the curve follows many small bumps of the training data, e.g., the waves before 1950, where there are only few houses per year. With the very large λ (green), it is almost a straight line. The medium λ (blue) lies in between. Let's compare them on the test set:

In [ ]:
print('Test RMSE (λ = 0.01):     ', root_mean_squared_error(y_test, smooth_small_lam.predict(X_test[['YearBuilt']])))
print('Test RMSE (λ = 100):      ', root_mean_squared_error(y_test, smooth_medium_lam.predict(X_test[['YearBuilt']])))
print('Test RMSE (λ = 1,000,000):', root_mean_squared_error(y_test, smooth_large_lam.predict(X_test[['YearBuilt']])))

The almost straight line (λ = 1,000,000) is clearly worse. The differences between the two other curves are small here: with only one predictor and many houses per year of construction, even the wiggly curve does not overfit much. λ controls the bias-variance trade-off (ISLP 7.5); for the GAM below, we will choose it with cross-validation.

## The GAM framework

In the notebook `ames_poly_steps_and_splines` and in the smoothing spline section above, we modelled a non-linear relationship for **one** predictor at a time: with polynomials, step functions, regression splines and smoothing splines. A generalized additive model (GAM, ISLP 7.7) combines such building blocks for **many** predictors:

$$\text{SalePrice} = \beta_0 + f_1(x_1) + f_2(x_2) + \dots + f_p(x_p) + \varepsilon$$

Each predictor gets its own function $f_j$, and the model simply **adds up** their contributions (hence *additive*). This has two advantages:

- Each $f_j$ can be non-linear, and we do not have to find the right shape (e.g., squares or knots) by hand.
- We can still look at each predictor separately: its $f_j$ shows how the predicted price changes with this predictor, holding the others fixed. This makes GAMs much easier to interpret than, e.g., neural networks.

The price for this: a GAM does not include interactions between predictors (unless we add them explicitly).

In pygam, the model is written as a sum of terms, one per predictor. In this notebook, every term is a **smoothing spline** `s(j)`, the building block from the section above (ISLP 7.5). The `j` is the **position** of the predictor's column (pygam does not understand column names). Our GAM uses five predictors: the year of construction (*YearBuilt*), the lot size (*LotArea*), the above-ground living area (*GrLivArea*), the basement area (*TotalBsmtSF*) and the overall quality rating (*OverallQual*, from 1 = very poor to 10 = very excellent). We first select them in a fixed order:

In [ ]:
gam_columns = ['YearBuilt', 'LotArea', 'GrLivArea', 'TotalBsmtSF', 'OverallQual']

X_train_gam = X_train[gam_columns]
X_test_gam = X_test[gam_columns]

X_train_gam.head()

### Fit a linear regression as the baseline

To assess the performance of the GAM, we first fit a simple baseline model: a linear regression with the same five predictors.

In [ ]:
mod_01 = smf.ols(formula='SalePrice ~ YearBuilt + LotArea + GrLivArea + TotalBsmtSF + OverallQual', data=pd.concat([X_train, y_train], axis=1))
mod_01 = mod_01.fit()
print(mod_01.summary2())

We evaluate the models by their RMSE on the training and on the test set.

In [ ]:
y_pred_train = mod_01.predict(X_train)
y_pred_test = mod_01.predict(X_test)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

### Fit a GAM

Our GAM uses a smoothing spline for each of the five predictors (positions 0 to 4):

`s(0) + s(1) + s(2) + s(3) + s(4)`

Each smoothing spline has a penalty λ (`lam`), which controls how wiggly it can be (ISLP 7.5.1). To start, we fit the GAM with almost no penalty (λ = 0.001).

In [ ]:
gam_wiggly = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4), lam=0.001)
gam_wiggly.fit(X_train_gam, y_train)

In [ ]:
y_pred_train = gam_wiggly.predict(X_train_gam)
y_pred_test = gam_wiggly.predict(X_test_gam)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

The training RMSE is much lower than that of the linear baseline, but the test RMSE is **higher**: the nearly unpenalized GAM overfits the training data.

Let's look at the patterns the model has learned. For each term, we draw its function $f_j$ (the *partial effect* on the predicted price), with a 95% confidence band (dashed), like ISLP Figures 7.11 and 7.12. The bands are wide where data are sparse. Note how wiggly some of the curves are.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, term in enumerate(gam_wiggly.terms):
    if term.isintercept:
        continue

    x_grid = gam_wiggly.generate_X_grid(term=i)
    pdep, confi = gam_wiggly.partial_dependence(term=i, X=x_grid, width=0.95)

    axes[i].plot(x_grid[:, term.feature], pdep)
    axes[i].plot(x_grid[:, term.feature], confi, c='r', ls='--')
    axes[i].set_xlabel(gam_columns[term.feature])
    axes[i].set_title(gam_columns[term.feature])

axes[0].set_ylabel('Partial effect on SalePrice')
axes[3].set_ylabel('Partial effect on SalePrice')
axes[5].axis('off')   # we only have five terms
plt.tight_layout()
plt.show()

## Tune the smoothness penalty

λ controls the bias-variance trade-off (ISLP 7.5): a small λ gives a flexible fit with low bias but high variance (overfitting), a large λ gives a smooth, almost linear fit with low variance but higher bias. We choose λ by 5-fold **cross-validation on the training data** (ISLP 5.1.3), so the test set stays untouched.

We try 8 values between $10^{-3}$ and $10^{4}$, evenly spaced on a log scale, and use the same λ for all terms.

In [ ]:
lams = np.logspace(-3, 4, 8)
lams

Define the folds: the training data are shuffled and split into 5 folds.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

For every λ, the loop fits the GAM five times, each time on four folds, and computes the RMSE on the held-out fold. We store the mean CV RMSE per λ in `results`.

In [ ]:
results = []

for lam in lams:
    fold_rmse = []
    for train_idx, val_idx in kf.split(X_train_gam):
        gam_cv = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4), lam=lam)
        gam_cv.fit(X_train_gam.iloc[train_idx], y_train.iloc[train_idx])
        y_pred_val = gam_cv.predict(X_train_gam.iloc[val_idx])
        fold_rmse.append(root_mean_squared_error(y_train.iloc[val_idx], y_pred_val))

    results.append({'lam': lam, 'cv_rmse': np.mean(fold_rmse)})

results_df = pd.DataFrame(results)
results_df

Plot the CV RMSE against λ (log scale).

In [ ]:
plt.plot(results_df['lam'], results_df['cv_rmse'], marker='o')
plt.xscale('log')
plt.xlabel('lambda (log scale)')
plt.ylabel('CV RMSE')
plt.show()

The curve is U-shaped: with a very small λ the GAM overfits (high variance), with a very large λ the curves become too smooth (high bias). We pick the λ with the lowest CV RMSE.

In [ ]:
best_lam = results_df.loc[results_df['cv_rmse'].idxmin(), 'lam']
best_lam

Refit the GAM with the best λ on all training data and evaluate it once on the test set.

In [ ]:
gam_tuned = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4), lam=best_lam)
gam_tuned.fit(X_train_gam, y_train)

In [ ]:
y_pred_train = gam_tuned.predict(X_train_gam)
y_pred_test = gam_tuned.predict(X_test_gam)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

The tuned GAM has a slightly higher training RMSE than the wiggly GAM, but a clearly lower test RMSE, which is also lower than that of the linear baseline: allowing (smooth) non-linear effects pays off here.

Finally, let's visualize the effect of the tuning on the learned relationships. Compared with the wiggly GAM, the curves for `YearBuilt` and `GrLivArea` are smoother and easier to read: prices rise with newer construction years and with more living area, and the quality rating matters most at the top: going from 8 to 10 adds much more to the price than going from 2 to 4. For `LotArea` and `TotalBsmtSF`, the curves still bend strongly at large values. There are only a few houses with huge lots or basements (see the wide confidence bands), so we should not trust these parts of the curves.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, term in enumerate(gam_tuned.terms):
    if term.isintercept:
        continue

    x_grid = gam_tuned.generate_X_grid(term=i)
    pdep, confi = gam_tuned.partial_dependence(term=i, X=x_grid, width=0.95)

    axes[i].plot(x_grid[:, term.feature], pdep)
    axes[i].plot(x_grid[:, term.feature], confi, c='r', ls='--')
    axes[i].set_xlabel(gam_columns[term.feature])
    axes[i].set_title(gam_columns[term.feature])

axes[0].set_ylabel('Partial effect on SalePrice')
axes[3].set_ylabel('Partial effect on SalePrice')
axes[5].axis('off')   # we only have five terms
plt.tight_layout()
plt.show()

## Your turn!

**Your task:**

1. Extend the GAM with further predictors (e.g., `GarageArea`, `YearRemodAdd`, `Fireplaces`). Add a smoothing spline `s()` for each new predictor. Tune λ again with cross-validation and compare the test RMSE with the models above.
2. The LotArea curve is sensitive to a few huge lots. Try to make it more robust, e.g., by fixing a larger λ for this term only (`s(1, lam=1000)`) or by using `np.log(LotArea)` as predictor. Does the test RMSE improve?

In [ ]:
# YOUR CODE HERE